In [1]:
!pip install phe

  Using cached phe-1.5.0-py2.py3-none-any.whl.metadata (3.8 kB)
Using cached phe-1.5.0-py2.py3-none-any.whl (53 kB)


In [ ]:
!pip install haversine

In [4]:
import math
from phe import paillier
from haversine import haversine

# Step 1: Key Generation (Alice)
public_key, private_key = paillier.generate_paillier_keypair()

# Example coordinates in degrees
lat_A = 50.379320  # Alice's latitude
lon_A = -4.131244  # Alice's longitude
lat_B = 50.381813  # Bob's latitude
lon_B = -4.127100  # Bob's longitude

# Define the geofence
geofence_radius = 0.41  # Radius in kilometers

# Convert degrees to radians
latA = math.radians(lat_A)
lonA = math.radians(lon_A)
latB = math.radians(lat_B)
lonB = math.radians(lon_B)

# Compute the trigonometric values as per the protocol
alpha = math.cos(latA / 2)
beta = math.sin(latB / 2)
gamma = math.sin(latA / 2)
delta = math.cos(latB / 2)
zeta = math.cos(latA)
eta = math.cos(latB)
theta = math.sin(lonA / 2)
lambda_ = math.cos(lonB / 2)
mu = math.cos(lonA / 2)
nu = math.sin(lonB / 2)

# Step 2: Alice computes encrypted values and sends them to Bob
alpha_squared = alpha**2
neg_two_alpha_gamma = -2 * alpha * gamma
gamma_squared = gamma**2
zeta_eta_theta_lambda_squared = zeta * eta * (theta**2) * (lambda_**2)
neg_two_zeta_eta_theta_lambda = -2 * zeta * eta * theta * lambda_
zeta_mu = zeta * mu**2

# Encrypt the values
enc_alpha_squared = public_key.encrypt(alpha_squared)
enc_neg_two_alpha_gamma = public_key.encrypt(neg_two_alpha_gamma)
enc_gamma_squared = public_key.encrypt(gamma_squared)
enc_zeta_eta_theta_lambda_squared = public_key.encrypt(zeta_eta_theta_lambda_squared)
enc_neg_two_zeta_eta_theta_lambda = public_key.encrypt(neg_two_zeta_eta_theta_lambda)
enc_zeta_mu = public_key.encrypt(zeta_mu)

# Alice sends encrypted values to Bob
alice_data = {
    "enc_alpha_squared": enc_alpha_squared,
    "enc_neg_two_alpha_gamma": enc_neg_two_alpha_gamma,
    "enc_gamma_squared": enc_gamma_squared,
    "enc_zeta_eta_theta_lambda_squared": enc_zeta_eta_theta_lambda_squared,
    "enc_neg_two_zeta_eta_theta_lambda": enc_neg_two_zeta_eta_theta_lambda,
    "enc_zeta_eta": enc_zeta_mu,
}

# Step 3: Bob computes JaK using homomorphic operations
beta_squared = beta**2
delta_squared = delta**2
mu_nu = mu * nu
eta_nu_squared = eta * nu**2

enc_a = (
    alice_data["enc_alpha_squared"] * beta_squared
    + alice_data["enc_neg_two_alpha_gamma"] * (beta * delta)
    + alice_data["enc_gamma_squared"] * delta_squared
    + alice_data["enc_zeta_eta_theta_lambda_squared"]
    + alice_data["enc_neg_two_zeta_eta_theta_lambda"] * mu_nu
    + alice_data["enc_zeta_eta"] * eta_nu_squared
)

# Bob sends enc_a back to Alice
# Step 4: Alice decrypts a and computes the distance
a = private_key.decrypt(enc_a)

# Earth's radius in kilometers
R = 6371.0

# Compute haversine distance
distance = 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# Determine if Alice's point is inside or outside the geofence
if distance <= geofence_radius:
    print(f"ALice's location is INSIDE the geofence (distance: {distance:.2f} km).")
else:
    print(f"Alice's location is OUTSIDE the geofence (distance: {distance:.2f} km).")

# Output the computed distance for verification
print(f"Privacy-preserving haversine distance: {distance:.2f} km")

# Validate using standard haversine library
haversine_distance = haversine((lat_A, lon_A), (lat_B, lon_B))
print(f"Distance between Alice and Bob (using haversine library): {haversine_distance:.2f} km")

ALice's location is INSIDE the geofence (distance: 0.40 km).
Privacy-preserving haversine distance: 0.40 km
Distance between Alice and Bob (using haversine library): 0.40 km
